# Part 2 — Depth, Open Interest and Liquidation Tests (Binance USD-M futures)
This notebook tests **T6 on the data it is actually about**: order-book depth. Part 1 used spot klines (price and trades only), which cannot see depth.

**What it does**
1. **D1** finds which futures archives exist (klines, `bookDepth`, open interest, liquidations) and their date ranges.
2. **D2** downloads every day, checks each file against Binance's published checksum, and cleans the depth data with fixed rules V1–V5. Binance's depth archive has known bad periods, such as misaligned prices in Apr–May 2025 and partial days.
3. **D3–D6** use **training days only (≤ 2024-12-31)**: they estimate the T6 model, test whether the price overshoot is carried by depth, and relate T6 to liquidations.
4. **D7** writes a **new pre-registration** before the 2025–2026 data are evaluated.
5. **D8** runs the **held-out test on 2025–2026**. That period has never been looked at; every evaluation is counted.
6. **D9** covers the event days, including Apr 2025 and Oct 2025.

**Before running:** keep `USE_DRIVE = True` (cell D0). The first run downloads about 4 years of daily files (roughly 30–90 min); later runs use the cache. `DAY_STEP = 2` halves the time.

**Rules:** do not change D7's spec after running D8. If you must, disclose it; the unlock counter records every evaluation. Whatever D8 shows goes in the paper.

In [ ]:
# ============================================================================
# CELL D0 — SETUP (self-contained; run first)
# Part 2 of the test suite: order-book depth, open interest and liquidations
# from Binance USD-M futures public archives. Writes to the same results/ folder
# as Part 1 when USE_DRIVE = True (recommended: these downloads take a while).
# ============================================================================
import os, sys, json, time, math, glob, re, io, zipfile, hashlib, subprocess, warnings, itertools
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed

IN_COLAB = "google.colab" in sys.modules
for _p, _m in [("pyarrow", "pyarrow")]:
    try:
        __import__(_m)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _p], check=True)

import numpy as np
import pandas as pd
from scipy import stats, linalg
warnings.filterwarnings("ignore", category=RuntimeWarning)

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ceiling_suite"
elif IN_COLAB:
    BASE = "/content/ceiling_suite"
else:
    BASE = os.path.abspath("ceiling_suite_local")
RESULTS, PREREG = os.path.join(BASE, "results"), os.path.join(BASE, "prereg")
FUTDATA = os.path.join(BASE, "data_futures")
for _d in (RESULTS, PREREG, FUTDATA):
    os.makedirs(_d, exist_ok=True)

SEED = 20261018
def rng(offset=0):
    return np.random.default_rng(SEED + offset)

def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

def record(test_id, passed, kind, metrics, criterion="", note=""):
    assert kind in {"theorem-check", "simulation", "real-data", "audit"}, kind
    status = "REPORT" if passed is None else ("PASS" if passed else "FAIL")
    out = {"test_id": test_id, "status": status, "kind": kind, "criterion": criterion, "note": note,
           "metrics": _clean(metrics), "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "seed": SEED}
    with open(os.path.join(RESULTS, f"{test_id}.json"), "w") as f:
        json.dump(out, f, indent=2)
    print(f"[{status}] {test_id} ({kind})" + (f" — {criterion}" if criterion else ""))
    if note:
        print("        note:", note)
    return out

def boot_ci(v, R, B=2000, stat=np.mean):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 3:
        return [np.nan, np.nan]
    bs = [stat(v[R.integers(0, len(v), len(v))]) for _ in range(B)]
    return [float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))]

# ---- Sample design (fixed BEFORE looking at any futures data) --------------
SYMBOL = "BTCUSDT"
TRAIN_END = "2024-12-31"        # exploration + estimation allowed on dates <= this
TEST_START = "2025-01-01"       # untouched: evaluated once, after pre-registration (cell D8)
DAY_STEP = 1                    # 1 = every day; 2 = every other day (halves download time)
EVENT_DAYS = {"SVB2023": "2023-03-11", "Aug2024": "2024-08-05",
              "Apr2025": "2025-04-07", "Oct2025": "2025-10-10"}
print("BASE =", BASE)


In [ ]:
# ============================================================================
# CELL D1 — DATA ACCESS AND INVENTORY (what actually exists on data.binance.vision)
# Datasets (USD-M futures, daily files):
#   klines 1m      price + signed taker flow
#   bookDepth      cumulative depth/notional at ±1..5% bands, ~30 s snapshots (undocumented format)
#   metrics        open interest (5-min)
#   liquidationSnapshot   forced orders (published only until 2024-03-31)
# This cell probes availability and finds the first/last date of each dataset.
# ============================================================================
ROOT_URL = "https://data.binance.vision/data/futures/um/daily"
def ds_url(ds, date, sym=SYMBOL):
    if ds == "klines":
        return f"{ROOT_URL}/klines/{sym}/1m/{sym}-1m-{date}.zip"
    return f"{ROOT_URL}/{ds}/{sym}/{sym}-{ds}-{date}.zip"

def _http(url, method="GET", timeout=60, tries=3):
    last = None
    for i in range(tries):
        try:
            req = urllib.request.Request(url, method=method, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.status, (r.read() if method == "GET" else b"")
        except urllib.error.HTTPError as e:
            if e.code in (403, 404):
                return e.code, b""
            last = e
        except Exception as e:
            last = e
        time.sleep(1.5 * (i + 1))
    raise RuntimeError(f"network error {url}: {last}")

def exists(ds, date):
    return _http(ds_url(ds, date), "HEAD", timeout=20)[0] == 200

def first_last_date(ds, anchors, lo="2019-09-01", hi=None):
    """First/last available daily file, by binary search on each side of a date known to exist.
    Assumes coverage is contiguous around the anchor (gaps are handled day by day later)."""
    hi = hi or time.strftime("%Y-%m-%d", time.gmtime(time.time() - 2 * 86400))
    anchor = next((d for d in anchors if exists(ds, d)), None)
    if anchor is None:
        return None, None
    left = pd.date_range(lo, anchor, freq="D").strftime("%Y-%m-%d").tolist()
    a, b = 0, len(left) - 1                  # first day that exists (left side: F..F T..T)
    while a < b:
        m = (a + b) // 2
        if exists(ds, left[m]):
            b = m
        else:
            a = m + 1
    right = pd.date_range(anchor, hi, freq="D").strftime("%Y-%m-%d").tolist()
    c, e = 0, len(right) - 1                 # last day that exists (right side: T..T F..F)
    while c < e:
        m = (c + e + 1) // 2
        if exists(ds, right[m]):
            c = m
        else:
            e = m - 1
    return left[a], right[c]

PROBE_DATES = ["2021-05-19", "2022-11-08", "2023-01-15", "2023-03-11", "2023-06-01", "2024-03-15",
               "2024-04-15", "2024-08-05", "2025-04-07", "2025-05-10", "2025-10-10", "2026-06-01"]
DATASETS = ["klines", "bookDepth", "metrics", "liquidationSnapshot"]
inv = pd.DataFrame({ds: [exists(ds, d) for d in PROBE_DATES] for ds in DATASETS}, index=PROBE_DATES)
print(inv.replace({True: "yes", False: "-"}).to_string())

COVERAGE = {ds: first_last_date(ds, [d for d in PROBE_DATES if inv.loc[d, ds]]) for ds in DATASETS}
for ds, (a, b) in COVERAGE.items():
    print(f"{ds:20s} first {a}  last {b}")
record("D1_inventory", None, "audit", {"probe": inv.astype(int).to_dict(), "coverage": COVERAGE},
       note="Coverage found by binary search; gaps inside a range are handled day by day later.")


In [ ]:
# ============================================================================
# CELL D2 — DOWNLOAD, VALIDATE AND BUILD THE 1-MINUTE PANEL
# Every archive is checked against its official .CHECKSUM when one is published.
# bookDepth validation rules (fixed here, before any analysis; also pre-registered in D7):
#   V1 a snapshot needs bands -5,-2,-1,+1,+2,+5 (present in both old and new formats)
#   V2 cumulative depth must be non-decreasing with distance from mid on each side
#   V3 implied average price notional/depth at ±1% must be within 3% of the 1-min futures close
#      (drops the known Apr–May 2025 misaligned snapshots and similar anomalies)
#   V4 runs of > 10 identical consecutive snapshots (stuck feed) are dropped
#   V5 a day is used only if >= 80% of its minutes have a valid depth snapshot
# ============================================================================
KL_COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time", "quote_volume",
           "count", "taker_buy_volume", "taker_buy_quote_volume", "ignore"]
DL_LOG = []

def fetch(ds, date):
    """Raw zip bytes (checksum-verified when possible) or None if the file does not exist."""
    url = ds_url(ds, date)
    code, raw = _http(url)
    if code != 200:
        return None
    sha = hashlib.sha256(raw).hexdigest()
    c2, chk = _http(url + ".CHECKSUM", timeout=30)
    verified = None
    if c2 == 200 and chk:
        official = chk.decode().split()[0]
        if official != sha:
            raise RuntimeError(f"SHA-256 mismatch {url}")
        verified = True
    DL_LOG.append({"ds": ds, "date": date, "sha256": sha, "checksum_verified": verified})
    return raw

def read_zip_csv(raw, default_cols=None):
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        name = [n for n in z.namelist() if n.endswith(".csv")][0]
        txt = z.read(name).decode("utf-8", errors="ignore")
    first = txt.split("\n", 1)[0]
    has_header = bool(re.search(r"[A-Za-z_]", first.split(",")[0]))
    df = pd.read_csv(io.StringIO(txt), header=0 if has_header else None)
    if not has_header and default_cols is not None:
        df.columns = default_cols[:df.shape[1]]
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

def to_utc_seconds(col):
    s = pd.Series(col)
    if pd.api.types.is_numeric_dtype(s):
        v = s.astype("int64").to_numpy()
        return np.where(v > 1e14, v // 1_000_000, np.where(v > 1e11, v // 1000, v))
    ts = pd.to_datetime(s, utc=True)                      # resolution-independent (pandas 2 and 3)
    return ((ts - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta("1s")).astype("int64").to_numpy()

def build_minute_day(date, need_liq):
    t0 = int(pd.Timestamp(f"{date} 00:00:00", tz="UTC").timestamp())
    idx = np.arange(1440)
    out = pd.DataFrame(index=idx)
    # ---- klines 1m ----
    raw = fetch("klines", date)
    if raw is None:
        return None, {"date": date, "reason": "no klines"}
    k = read_zip_csv(raw, KL_COLS)
    tk = (to_utc_seconds(k["open_time"]) - t0) // 60
    k = k.assign(m=tk).drop_duplicates("m").set_index("m").reindex(idx)
    out["close"] = k["close"].astype(float).ffill().bfill()
    out["flow"] = (2 * k["taker_buy_quote_volume"].astype(float) - k["quote_volume"].astype(float)).fillna(0) / 1e6
    out["vol"] = k["quote_volume"].astype(float).fillna(0) / 1e6
    stats_ = {"date": date}
    # ---- bookDepth ----
    raw = fetch("bookDepth", date)
    for c in ["bid1", "ask1", "bid2", "ask2", "bid5", "ask5"]:
        out[c] = np.nan
    if raw is not None:
        b = read_zip_csv(raw)
        b["sec"] = to_utc_seconds(b["timestamp"])
        b["percentage"] = pd.to_numeric(b["percentage"], errors="coerce").round().astype("Int64")
        need = [-5, -2, -1, 1, 2, 5]
        piv_n = b.pivot_table(index="sec", columns="percentage", values="notional", aggfunc="last")
        piv_d = b.pivot_table(index="sec", columns="percentage", values="depth", aggfunc="last")
        n0 = len(piv_n)
        ok = piv_n.reindex(columns=need).notna().all(axis=1) & piv_d.reindex(columns=need).notna().all(axis=1)       # V1
        piv_n, piv_d = piv_n[ok], piv_d[ok]
        mono = ((piv_d[-2] >= piv_d[-1]) & (piv_d[-5] >= piv_d[-2]) &
                (piv_d[2] >= piv_d[1]) & (piv_d[5] >= piv_d[2]))                                         # V2
        minute = ((piv_n.index.to_numpy() - t0) // 60).clip(0, 1439)
        ref = out["close"].to_numpy()[minute]
        imp_b, imp_a = piv_n[-1] / piv_d[-1], piv_n[1] / piv_d[1]
        price_ok = (np.abs(imp_b / ref - 1) < 0.03) & (np.abs(imp_a / ref - 1) < 0.03)                   # V3
        sig = piv_n[need].round(2).astype(str).agg("|".join, axis=1)
        runid = (sig != sig.shift()).cumsum()
        runlen = sig.groupby(runid).transform("size")
        not_stale = runlen <= 10                                                                         # V4
        good = mono & price_ok & not_stale
        g = piv_n[good]
        gm = ((g.index.to_numpy() - t0) // 60).clip(0, 1439)
        snap = pd.DataFrame({"m": gm, "bid1": g[-1].to_numpy(), "ask1": g[1].to_numpy(),
                             "bid2": g[-2].to_numpy(), "ask2": g[2].to_numpy(),
                             "bid5": g[-5].to_numpy(), "ask5": g[5].to_numpy()}).groupby("m").last() / 1e6
        snap = snap.reindex(idx).ffill(limit=2)
        for c in snap.columns:
            out[c] = snap[c]
        stats_.update({"snapshots": n0, "drop_V1": int((~ok).sum()), "drop_V2": int((~mono).sum()),
                       "drop_V3": int((~price_ok).sum()), "drop_V4": int((~not_stale).sum()),
                       "minute_coverage": float(out["bid1"].notna().mean())})
    else:
        stats_.update({"snapshots": 0, "minute_coverage": 0.0})
    stats_["depth_day_ok"] = stats_["minute_coverage"] >= 0.80                                          # V5
    # ---- open interest (metrics, 5-min) ----
    raw = fetch("metrics", date)
    out["oi"] = np.nan
    if raw is not None:
        mt = read_zip_csv(raw)
        tcol = "create_time" if "create_time" in mt.columns else mt.columns[0]
        mm = ((to_utc_seconds(mt[tcol]) - t0) // 60).clip(0, 1439)
        oi = pd.Series(pd.to_numeric(mt["sum_open_interest_value"], errors="coerce").to_numpy() / 1e6, index=mm)
        out["oi"] = oi.groupby(level=0).last().reindex(idx).ffill().bfill()
    # ---- liquidations (until 2024-03-31) ----
    out["liq_long"] = np.nan; out["liq_short"] = np.nan
    if need_liq:
        raw = fetch("liquidationSnapshot", date)
        if raw is not None:
            lq = read_zip_csv(raw)
            tcol = "time" if "time" in lq.columns else lq.columns[0]
            px = pd.to_numeric(lq.get("average_price", lq.get("price")), errors="coerce")
            qty = pd.to_numeric(lq.get("accumulated_fill_quantity", lq.get("original_quantity")), errors="coerce")
            notional = (px * qty).fillna(0) / 1e6
            lm = ((to_utc_seconds(lq[tcol]) - t0) // 60).clip(0, 1439)
            side = lq["side"].astype(str).str.upper()
            out["liq_long"] = pd.Series(np.where(side == "SELL", notional, 0.0), index=lm).groupby(level=0).sum().reindex(idx).fillna(0)
            out["liq_short"] = pd.Series(np.where(side == "BUY", notional, 0.0), index=lm).groupby(level=0).sum().reindex(idx).fillna(0)
    out["date"] = date
    return out, stats_

MIN_DIR = os.path.join(FUTDATA, "minute"); os.makedirs(MIN_DIR, exist_ok=True)

def get_day(date):
    pq, js = os.path.join(MIN_DIR, f"{date}.parquet"), os.path.join(MIN_DIR, f"{date}.json")
    if os.path.exists(pq) and os.path.exists(js):
        return pd.read_parquet(pq), json.load(open(js))
    liq_last = COVERAGE.get("liquidationSnapshot", (None, None))[1]
    df, st = build_minute_day(date, need_liq=bool(liq_last and date <= liq_last))
    if df is not None:
        df.to_parquet(pq)
    json.dump(_clean(st), open(js, "w"))
    return df, st

dep_first, dep_last = COVERAGE["bookDepth"]
assert dep_first, "bookDepth not found on data.binance.vision — see D1 output"
ALL_DATES = pd.date_range(dep_first, dep_last, freq="D").strftime("%Y-%m-%d").tolist()[::DAY_STEP]
for d in EVENT_DAYS.values():          # always include the event days
    if dep_first <= d <= dep_last and d not in ALL_DATES:
        ALL_DATES.append(d)
ALL_DATES = sorted(ALL_DATES)
print(f"building {len(ALL_DATES)} days from {ALL_DATES[0]} to {ALL_DATES[-1]} (cached days are skipped)")

DAYS, VSTATS = {}, []
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = {ex.submit(get_day, d): d for d in ALL_DATES}
    for i, f in enumerate(as_completed(futs), 1):
        d = futs[f]
        try:
            df, st = f.result()
            VSTATS.append(st)
            if df is not None:
                DAYS[d] = df
        except Exception as e:
            VSTATS.append({"date": d, "reason": f"error: {e}"})
        if i % 50 == 0:
            print(f"   {i}/{len(ALL_DATES)} days")
vst = pd.DataFrame(VSTATS).sort_values("date")
if "depth_day_ok" not in vst:
    vst["depth_day_ok"] = False
vst["depth_day_ok"] = vst["depth_day_ok"].fillna(False).astype(bool)
errs = vst[vst.get("reason", pd.Series(index=vst.index, dtype=object)).astype(str).str.startswith("error")] if "reason" in vst else vst.iloc[:0]
if len(errs):
    print(f"!! {len(errs)} days failed with errors, e.g.:", errs["reason"].iloc[0])
ok_days = sorted(vst.loc[vst["depth_day_ok"], "date"])
print(f"days built: {len(DAYS)}; days passing depth validation: {len(ok_days)}")
pd.DataFrame(DL_LOG).to_csv(os.path.join(RESULTS, "futures_manifest_sha256.csv"), index=False)
vsum = {k: int(vst[k].fillna(0).sum()) for k in ["snapshots", "drop_V1", "drop_V2", "drop_V3", "drop_V4"] if k in vst}
record("D2_depth_validation", None, "audit",
       {"days_requested": len(ALL_DATES), "days_built": len(DAYS), "days_valid_depth": len(ok_days),
        "snapshot_drops": vsum,
        "failed_days": vst.loc[~vst["depth_day_ok"], "date"].tolist()[:200], "n_error_days": int(len(errs))},
       note="Report these counts in the data section. V3 drops show where Binance's depth archive is misaligned.")


## Training-period analysis (≤ 2024-12-31): exploration and estimation are allowed here

In [ ]:
# ============================================================================
# CELL D3 — ESTIMATE THE T6 MODEL ON REAL DEPTH DATA (training days only)
# 1-minute bars.  p = log price (bp), u = signed taker flow ($1M),
# d = (bid1 - ask1)/(bid1 + ask1) = depth imbalance within 1% of mid (unit-free),
# x = p - EMA_60min(p).
#   price:    Δp_t = c + κ u_t + φ d_{t-1} - (1-a) x_{t-1} + e     (Δp on the left: no EMA mechanics)
#   depth:    d_t  = c + (1-r) d_{t-1} + ε u_t + e
#   momentum: u_t  = c + (1-η) u_{t-1} + ηq x_{t-1} + e
# T6 conditions (bp per $1M):  C2 = φε - (1-a)κ ;  C3 = φε + κ²ηq - (1-a)κ.   Amplification <=> C > 0.
# Also the implied impulse response: peak ratio max_k h_k / h_0 and peak minute.
# These are reduced-form estimates (u_t is contemporaneous); label them so in the paper.
# ============================================================================
def prep(df):
    p = np.log(df["close"].to_numpy()) * 1e4
    u = df["flow"].to_numpy()
    tot = (df["bid1"] + df["ask1"]).to_numpy()
    d = ((df["bid1"] - df["ask1"]) / (df["bid1"] + df["ask1"])).to_numpy()
    x = p - pd.Series(p).ewm(span=60, adjust=False).mean().to_numpy()
    return p, u, d, x, tot

def ols(y, X):
    ok = np.all(np.isfinite(X), 1) & np.isfinite(y)
    if ok.sum() < X.shape[1] + 20:
        return None
    X1 = np.column_stack([np.ones(ok.sum()), X[ok]])
    return np.linalg.lstsq(X1, y[ok], rcond=None)[0][1:]

def fit_t6(p, u, d, x):
    dp = np.diff(p, prepend=np.nan)
    bp = ols(dp[1:], np.column_stack([u[1:], d[:-1], x[:-1]]))
    bd = ols(d[1:], np.column_stack([d[:-1], u[1:]]))
    bm = ols(u[1:], np.column_stack([u[:-1], x[:-1]]))
    if bp is None or bd is None or bm is None:
        return None
    kappa, phi, a = bp[0], bp[1], 1 + bp[2]
    rho_d, eps = bd[0], bd[1]
    eta = 1 - bm[0]
    q = bm[1] / eta if abs(eta) > 1e-6 else np.nan
    C2 = phi * eps - (1 - a) * kappa
    C3 = phi * eps + kappa ** 2 * eta * q - (1 - a) * kappa
    h = impulse_t6(a, phi, kappa, 1 - rho_d, eps, eta, q)
    ok = np.all(np.isfinite(h)) and abs(h[0]) > 1e-12
    return {"kappa": kappa, "phi": phi, "a": a, "r": 1 - rho_d, "eps": eps, "eta": eta, "q": q,
            "C2": C2, "C3": C3,
            "peak_ratio": float(np.max(h) / h[0]) if ok else np.nan,
            "peak_minute": int(np.argmax(h)) if ok else -1}

def impulse_t6(a, phi, kappa, r, eps, eta, q, L=120):
    x = np.zeros(L); dd = np.zeros(L); f = np.zeros(L)
    for t in range(L):
        if t > 0:
            f[t] = (1 - eta) * f[t - 1] + eta * q * x[t - 1]
        tot = (1.0 if t == 0 else 0.0) + f[t]
        x[t] = (a * x[t - 1] + phi * dd[t - 1] if t else 0) + kappa * tot
        dd[t] = ((1 - r) * dd[t - 1] if t else 0) + eps * tot
        if abs(x[t]) > 1e9:
            return np.full(L, np.nan)
    return x

train_days = [d for d in ok_days if d <= TRAIN_END]
rows = []
for d in train_days:
    p, u, dd_, x, tot = prep(DAYS[d])
    est = fit_t6(p, u, dd_, x)
    if est:
        est.update({"date": d, "rv": float(np.nanstd(np.diff(p))), "depth1_musd": float(np.nanmedian(tot))})
        rows.append(est)
d3 = pd.DataFrame(rows)
hi = d3["rv"] >= d3["rv"].quantile(0.9)
R3 = rng(33)
summ = {}
for lab, sub in [("all_train_days", d3), ("high_vol_days(top10%)", d3[hi]), ("other_days", d3[~hi])]:
    summ[lab] = {"n_days": len(sub),
                 **{f"median_{k}": float(sub[k].median()) for k in ["kappa", "phi", "a", "r", "eps", "eta", "q", "C2", "C3", "peak_ratio"]},
                 "mean_C3": float(sub["C3"].mean()), "mean_C3_ci": boot_ci(sub["C3"], R3),
                 "share_C3_pos": float((sub["C3"] > 0).mean()), "share_C2_pos": float((sub["C2"] > 0).mean()),
                 "median_peak_minute": float(sub["peak_minute"].median())}
ev = {k: d3.loc[d3["date"] == v].drop(columns=["date"]).to_dict("records") for k, v in EVENT_DAYS.items()
      if (d3["date"] == v).any()}
print(pd.DataFrame(summ).T.round(4).to_string())
print("event days:", json.dumps(_clean(ev))[:800])
record("D3_T6_structural_train", None, "real-data", {"summary": summ, "event_days": ev, "n_days": len(d3)},
       note="C3 > 0 means the estimated market sits in T6's amplification region. Compare high-vol days with others.")


In [ ]:
# ============================================================================
# CELL D4 — IS THE PRICE OVERSHOOT CARRIED BY DEPTH? (mediation local projections, training days)
# y_k = p_{t+k} - p_{t-1} on u_t, with 5 lags of Δp, u, d as controls. Three versions:
#   (A) total
#   (B) + cumulative future flow u_{t+1..t+k}           (removes order-splitting persistence)
#   (C) + future flow + cumulative depth imbalance d_t + ... + d_{t+k-1}   (removes the depth channel:
#       in T6 the price path accumulates φ·Σd)
# T6 predicts: overshoot in (B) (max_{k>=1} β_k - β_0 > 0) that shrinks in (C).
# ============================================================================
H_MIN = [0, 1, 2, 3, 5, 10, 15, 30]

def lp_day_depth(df, nl=5):
    p, u, dd_, x, tot = prep(df)
    dp = np.diff(p, prepend=p[0])
    n = len(p); mk = max(H_MIN)
    t = np.arange(nl + 1, n - mk - 1)
    if not np.isfinite(dd_[t]).mean() > 0.9:
        return None
    L = np.column_stack([dp[t - j] for j in range(1, nl + 1)] + [u[t - j] for j in range(1, nl + 1)] +
                        [dd_[t - j] for j in range(1, nl + 1)])
    cu = np.concatenate([[0.0], np.cumsum(u)])
    cd = np.concatenate([[0.0], np.cumsum(np.nan_to_num(dd_))])
    out = {"A": [], "B": [], "C": []}
    for k in H_MIN:
        y = p[t + k] - p[t - 1]
        XA = np.column_stack([u[t], L])
        fut = cu[t + k + 1] - cu[t + 1]
        XB = np.column_stack([XA, fut]) if k > 0 else XA
        XC = np.column_stack([XB, cd[t + k] - cd[t]]) if k > 0 else XB
        for key, X in (("A", XA), ("B", XB), ("C", XC)):
            b = ols(y, X)
            out[key].append(b[0] if b is not None else np.nan)
    return {k: np.array(v) for k, v in out.items()}

per = [r for r in (lp_day_depth(DAYS[d]) for d in train_days) if r is not None]
R4 = rng(44); res4 = {}
for key in ("A", "B", "C"):
    M = np.array([r[key] for r in per]); M = M[np.all(np.isfinite(M), 1)]
    mean = M.mean(0)
    boots = np.array([M[R4.integers(0, len(M), len(M))].mean(0) for _ in range(1000)])
    over = boots[:, 1:].max(1) - boots[:, 0]
    res4[key] = {"beta_bp_per_musd": mean, "overshoot": float(mean[1:].max() - mean[0]),
                 "overshoot_ci": [float(np.quantile(over, .025)), float(np.quantile(over, .975))],
                 "peak_minute": H_MIN[int(np.argmax(mean))]}
    print(f"({key}) β by minute {np.round(mean, 3)}  overshoot {res4[key]['overshoot']:.3f} CI {np.round(res4[key]['overshoot_ci'], 3)}")
med = (res4["B"]["overshoot"] - res4["C"]["overshoot"]) / res4["B"]["overshoot"] if res4["B"]["overshoot"] > 0 else np.nan
record("D4_depth_mediation_train", None, "real-data", {"horizons_min": H_MIN, "versions": res4,
       "share_of_overshoot_carried_by_depth": med, "n_days": len(per)},
       note="Evidence for T6: overshoot CI > 0 in (B) and a clearly smaller overshoot in (C).")


In [ ]:
# ============================================================================
# CELL D5 — HOURLY BLOCKS: causal features and future outcomes
# For each day and hour t = 6..23 (minute 60h): features use minutes [t-360, t) only.
#   candidate (primary):  C3_hat from the rolling 6-hour T6 estimate (winsorized at train 1%/99%)
#   candidate (secondary): peak_ratio_hat
#   controls: rv_60, rv_360, abs_ret_60, flow_imb_60, log_vol_60, log_depth1, imbalance, oi_chg_60
#   outcomes over (t, t+60]: MAE_60 (bp), OI_drop_60 (%), depth_collapse_60 (%), liq_60 ($M, until 2024-03)
# ============================================================================
def block_features(df, t):
    p, u, dd_, x, tot = prep(df)
    w = slice(t - 360, t); h = slice(t - 60, t)
    est = fit_t6(p[w], u[w], dd_[w], x[w])
    r60 = np.diff(p[h]); r360 = np.diff(p[w])
    oi = df["oi"].to_numpy()
    return {"C3_hat": est["C3"] if est else np.nan,
            "peak_ratio_hat": est["peak_ratio"] if est else np.nan,
            "rv_60": np.nanstd(r60), "rv_360": np.nanstd(r360),
            "abs_ret_60": abs(p[t - 1] - p[t - 60]),
            "flow_imb_60": np.nansum(u[h]) / (np.nansum(np.abs(u[h])) + 1e-12),
            "log_vol_60": np.log1p(np.nansum(df["vol"].to_numpy()[h])),
            "log_depth1": np.log(tot[t - 1]) if np.isfinite(tot[t - 1]) and tot[t - 1] > 0 else np.nan,
            "imbalance": dd_[t - 1],
            "oi_chg_60": (oi[t - 1] / oi[t - 60] - 1) * 100 if np.isfinite(oi[t - 60]) and oi[t - 60] > 0 else np.nan}

def block_outcomes(df, t):
    p, u, dd_, x, tot = prep(df)
    f = slice(t, min(t + 61, len(p)))
    oi = df["oi"].to_numpy(); liq = (df["liq_long"] + df["liq_short"]).to_numpy()
    return {"MAE_60": float(np.nanmax(np.abs(p[f] - p[t - 1]))),
            "OI_drop_60": float((oi[t - 1] - np.nanmin(oi[f])) / oi[t - 1] * 100) if oi[t - 1] > 0 else np.nan,
            "depth_collapse_60": float((1 - np.nanmin(tot[f]) / tot[t - 1]) * 100) if tot[t - 1] > 0 else np.nan,
            "liq_60": float(np.nansum(liq[t:t + 60])) if np.isfinite(liq[t:t + 60]).any() else np.nan}

BLK_PATH = os.path.join(FUTDATA, f"blocks_step{DAY_STEP}.parquet")
if os.path.exists(BLK_PATH) and set(pd.read_parquet(BLK_PATH)["date"]) >= set(ok_days):
    blocks = pd.read_parquet(BLK_PATH)
else:
    rows = []
    for i, d in enumerate(ok_days):
        df = DAYS[d]
        for hr in range(6, 24):
            t = hr * 60
            if t + 1 >= 1440:
                continue
            r_ = block_features(df, t); r_.update(block_outcomes(df, t)); r_.update({"date": d, "hour": hr})
            rows.append(r_)
        if (i + 1) % 100 == 0:
            print(f"   blocks: {i+1}/{len(ok_days)} days")
    blocks = pd.DataFrame(rows)
    blocks.to_parquet(BLK_PATH)
blocks["split"] = np.where(blocks["date"] <= TRAIN_END, "train", np.where(blocks["date"] >= TEST_START, "test", "gap"))
print(blocks.groupby("split").size().to_dict())

# leakage test: scramble everything from t on, features must not change
Rl = rng(55); fails = 0
for _ in range(30):
    d = Rl.choice(ok_days); t = int(Rl.integers(6, 23)) * 60
    a_ = block_features(DAYS[d], t)
    df2 = DAYS[d].copy()
    for c in ["close", "flow", "vol", "bid1", "ask1", "oi"]:
        v = df2[c].to_numpy().copy(); v[t:] = v[t:][::-1] * 1.7; df2[c] = v
    b_ = block_features(df2, t)
    fails += any((np.isfinite(a_[k]) or np.isfinite(b_[k])) and not np.isclose(a_[k], b_[k], equal_nan=True) for k in a_)
record("D5_no_lookahead", fails == 0, "audit", {"trials": 30, "failures": fails},
       criterion="hourly features unchanged when all data from t onward are scrambled")


In [ ]:
# ============================================================================
# CELL D6 — DO T6 CONDITIONS PRECEDE LIQUIDATIONS? (training period, in-sample; liquidation data
# exist only until 2024-03-31). Reported as descriptive evidence, not as the held-out test.
# ============================================================================
lq = blocks[(blocks["split"] == "train") & blocks["liq_60"].notna()].copy()
res6 = {"n_blocks": int(len(lq))}
if len(lq) > 200:
    lq["log_liq"] = np.log1p(lq["liq_60"])
    lo, hi_ = lq["C3_hat"].quantile([0.01, 0.99])
    lq["C3w"] = lq["C3_hat"].clip(lo, hi_)
    ctrl = ["rv_60", "rv_360", "abs_ret_60", "flow_imb_60", "log_vol_60", "log_depth1", "imbalance", "oi_chg_60"]
    sub = lq.dropna(subset=ctrl + ["C3w", "log_liq"])
    X0 = np.column_stack([np.ones(len(sub)), sub[ctrl].to_numpy()])
    X1 = np.column_stack([X0, sub["C3w"].to_numpy()])
    y = sub["log_liq"].to_numpy()
    b0 = np.linalg.lstsq(X0, y, rcond=None)[0]; b1 = np.linalg.lstsq(X1, y, rcond=None)[0]
    r2_0 = 1 - np.sum((y - X0 @ b0) ** 2) / np.sum((y - y.mean()) ** 2)
    r2_1 = 1 - np.sum((y - X1 @ b1) ** 2) / np.sum((y - y.mean()) ** 2)
    # day-cluster bootstrap for the C3 coefficient
    R6 = rng(66); days_ = sub["date"].to_numpy(); ud_ = np.unique(days_); bs = []
    for _ in range(500):
        pick = R6.choice(ud_, len(ud_)); idx = np.concatenate([np.where(days_ == d)[0] for d in pick])
        bs.append(np.linalg.lstsq(X1[idx], y[idx], rcond=None)[0][-1])
    top = sub["C3w"] >= sub["C3w"].quantile(0.9)
    res6.update({"coef_C3": float(b1[-1]), "coef_C3_ci": [float(np.quantile(bs, .025)), float(np.quantile(bs, .975))],
                 "r2_controls": float(r2_0), "r2_with_C3": float(r2_1),
                 "mean_liq_musd_top_decile_C3": float(sub.loc[top, "liq_60"].mean()),
                 "mean_liq_musd_other": float(sub.loc[~top, "liq_60"].mean()),
                 "share_C3_positive": float((sub["C3_hat"] > 0).mean())})
print(json.dumps(_clean(res6), indent=1))
record("D6_liquidations_in_sample", None, "real-data", res6,
       note="In-sample only (data end 2024-03-31). Support for T6 = positive C3 coefficient with CI above 0.")


## Held-out test (2025–2026): pre-register first, then evaluate once

In [ ]:
# ============================================================================
# CELL D7 — PRE-REGISTRATION OF THE HELD-OUT DEPTH TEST (run BEFORE D8; commit the file to GitHub)
# ============================================================================
PREREG_DEPTH = {
    "question": "Does the T6 amplification statistic C3_hat, estimated from the previous 6 hours of "
                "futures depth, flow and price, predict the next hour's maximum adverse excursion beyond controls?",
    "data": "Binance USD-M BTCUSDT daily klines(1m), bookDepth, metrics; validation rules V1-V5 of cell D2",
    "unit": "hourly blocks, hours 6-23 UTC; features from minutes [t-360, t); outcomes from (t, t+60]",
    "primary_outcome": "MAE_60 (bp)", "secondary_outcomes": ["OI_drop_60", "depth_collapse_60"],
    "controls": ["rv_60", "rv_360", "abs_ret_60", "flow_imb_60", "log_vol_60", "log_depth1", "imbalance", "oi_chg_60"],
    "candidate": "C3_hat winsorized at train 1%/99%", "secondary_candidate": "peak_ratio_hat (same winsorizing)",
    "train": f"days <= {TRAIN_END}", "test": f"days >= {TEST_START}",
    "model": "OLS; tail = outcome above train 90th percentile; score = fitted value",
    "inference": "Clark-West (Newey-West 6 lags, blocks in time order); day-block bootstrap 2,000 for ΔR², ΔAUC",
    "success": "primary: Clark-West one-sided p < 0.05 AND bootstrap 95% CI of ΔR² excludes 0",
    "multiple_testing": "secondary outcomes/candidate reported with Holm-adjusted p-values",
}
spec_txt = json.dumps(PREREG_DEPTH, indent=2, sort_keys=True)
DEPTH_HASH = hashlib.sha256(spec_txt.encode()).hexdigest()
pd_path = os.path.join(PREREG, "P2_depth.json")
if not os.path.exists(pd_path):
    open(pd_path, "w").write(spec_txt); print("pre-registration written:", pd_path)
elif hashlib.sha256(open(pd_path).read().encode()).hexdigest() != DEPTH_HASH:
    print("!! WARNING: spec differs from the registered file. Disclose any change made after seeing results.")
print("spec SHA-256:", DEPTH_HASH)


In [ ]:
# ============================================================================
# CELL D8 — HELD-OUT TEST ON 2025-2026 (gate G2-depth). Every evaluation is counted.
# ============================================================================
CTRL_D = PREREG_DEPTH["controls"]
trD = blocks[blocks["split"] == "train"].copy(); teD = blocks[blocks["split"] == "test"].copy()
for cand in ["C3_hat", "peak_ratio_hat"]:
    lo, hi_ = trD[cand].quantile([0.01, 0.99])
    trD[cand + "_w"] = trD[cand].clip(lo, hi_); teD[cand + "_w"] = teD[cand].clip(lo, hi_)

def unlock_depth():
    pth = os.path.join(PREREG, "test_set_unlocks_depth.json")
    n = json.load(open(pth))["n"] + 1 if os.path.exists(pth) else 1
    json.dump({"n": n, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "spec_sha256": DEPTH_HASH}, open(pth, "w"))
    return n

def ols_fit_pred(cols, A, B, y):
    Xa = np.column_stack([np.ones(len(A)), A[cols].to_numpy()])
    b = np.linalg.lstsq(Xa, A[y].to_numpy(), rcond=None)[0]
    return np.column_stack([np.ones(len(B)), B[cols].to_numpy()]) @ b

def auc_(y, s):
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    rk = stats.rankdata(np.concatenate([pos, neg]))
    return (rk[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

def cw_test(y, fs, fb, lags=6):
    adj = (y - fs) ** 2 - ((y - fb) ** 2 - (fs - fb) ** 2)
    m = adj.mean(); e = adj - m; n = len(adj)
    lrv = e @ e / n + 2 * sum((1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n for l in range(1, lags + 1))
    t_ = m / np.sqrt(lrv / n)
    return float(t_), float(1 - stats.norm.cdf(t_))

def eval_depth(outcome, cand):
    cols_all = CTRL_D + [cand, outcome]
    A = trD.dropna(subset=cols_all); B = teD.dropna(subset=cols_all).sort_values(["date", "hour"])
    y = B[outcome].to_numpy()
    fb = ols_fit_pred(CTRL_D, A, B, outcome); ff = ols_fit_pred(CTRL_D + [cand], A, B, outcome)
    sst = np.sum((y - A[outcome].mean()) ** 2)
    r2b, r2f = 1 - np.sum((y - fb) ** 2) / sst, 1 - np.sum((y - ff) ** 2) / sst
    t_, p_ = cw_test(y, fb, ff)
    tail = (y > A[outcome].quantile(0.9)).astype(int)
    Rb = rng(88); dd = B["date"].to_numpy(); ud_ = np.unique(dd); dR2, dA = [], []
    for _ in range(2000):
        pick = Rb.choice(ud_, len(ud_)); idx = np.concatenate([np.where(dd == d)[0] for d in pick])
        s_ = np.sum((y[idx] - A[outcome].mean()) ** 2)
        dR2.append((np.sum((y[idx] - fb[idx]) ** 2) - np.sum((y[idx] - ff[idx]) ** 2)) / s_)
        dA.append(auc_(tail[idx], ff[idx]) - auc_(tail[idx], fb[idx]))
    return {"n_train": len(A), "n_test": len(B), "r2_base": r2b, "r2_full": r2f, "delta_r2": r2f - r2b,
            "delta_r2_ci": [float(np.nanquantile(dR2, .025)), float(np.nanquantile(dR2, .975))],
            "auc_base": auc_(tail, fb), "auc_full": auc_(tail, ff),
            "delta_auc_ci": [float(np.nanquantile(dA, .025)), float(np.nanquantile(dA, .975))],
            "clark_west_t": t_, "clark_west_p": p_}

assert os.path.exists(pd_path), "run D7 first"
n_unl = unlock_depth()
primary = eval_depth("MAE_60", "C3_hat_w")
secondary = {f"{o}|{c}": eval_depth(o, c) for o in ["MAE_60", "OI_drop_60", "depth_collapse_60"]
             for c in ["C3_hat_w", "peak_ratio_hat_w"] if not (o == "MAE_60" and c == "C3_hat_w")}
ps = sorted([(v["clark_west_p"], k) for k, v in secondary.items()])
holm = {k: min(1.0, p * (len(ps) - i)) for i, (p, k) in enumerate(ps)}
for k in secondary:
    secondary[k]["holm_p"] = holm[k]
print(json.dumps(_clean(primary), indent=1))
print(pd.DataFrame(secondary).T[["delta_r2", "clark_west_p", "holm_p"]].round(5).to_string())
g2d = primary["clark_west_p"] < 0.05 and primary["delta_r2_ci"][0] > 0
record("D8_depth_test_G2", bool(g2d), "real-data",
       {"primary": primary, "secondary": secondary, "prereg_sha256": DEPTH_HASH, "test_set_unlocks": n_unl},
       criterion=PREREG_DEPTH["success"],
       note="Held-out 2025-2026. Report whatever it shows; test_set_unlocks > 1 must be disclosed.")

# Volatility flag vs T6 flag on the test set (each flags 10% of train blocks)
A = trD.dropna(subset=["C3_hat_w", "rv_60", "MAE_60"]); B = teD.dropna(subset=["C3_hat_w", "rv_60", "MAE_60"])
tail = (B["MAE_60"] > A["MAE_60"].quantile(0.9)).to_numpy()
fw = (B["C3_hat_w"] > A["C3_hat_w"].quantile(0.9)).to_numpy(); fv = (B["rv_60"] > A["rv_60"].quantile(0.9)).to_numpy()
def pr_(f):
    tp = np.sum(f & tail); return {"flag_rate": f.mean(), "precision": tp / max(f.sum(), 1), "recall": tp / max(tail.sum(), 1)}
record("D8b_flag_comparison", None, "real-data", {"T6_flag": pr_(fw), "vol_flag": pr_(fv), "either": pr_(fw | fv),
       "n_tail_events": int(tail.sum())}, note="Does the T6 flag catch tail hours that a volatility flag misses?")


In [ ]:
# ============================================================================
# CELL D9 — EVENT STUDIES (SVB 2023, Aug 2024, Apr 2025, Oct 2025 where data exist)
# For each event day: hourly C3_hat, MAE_60, OI drop; percentile of C3_hat in the hours before
# the worst hour, relative to all training blocks. Descriptive; the Oct 2025 and Apr 2025
# events are in the held-out period.
# ============================================================================
trC = blocks.loc[blocks["split"] == "train", "C3_hat"].dropna().to_numpy()
evres = {}
for name, d in EVENT_DAYS.items():
    b = blocks[blocks["date"] == d].sort_values("hour")
    if b.empty:
        evres[name] = "no valid depth data for this day"
        continue
    worst = int(b.loc[b["MAE_60"].idxmax(), "hour"])
    before = b[(b["hour"] < worst) & (b["hour"] >= worst - 3)]
    evres[name] = {"date": d, "worst_hour_utc": worst, "worst_MAE_60_bp": float(b["MAE_60"].max()),
                   "C3_hat_by_hour": dict(zip(b["hour"].astype(int), b["C3_hat"].round(4))),
                   "C3_percentile_3h_before_worst": [float(stats.percentileofscore(trC, v)) for v in before["C3_hat"].dropna()],
                   "OI_drop_worst_hour_pct": float(b.loc[b["hour"] == worst, "OI_drop_60"].iloc[0])}
print(json.dumps(_clean(evres), indent=1)[:3000])
record("D9_event_studies", None, "real-data", evres,
       note="Support for T6 = C3_hat in a high percentile in the hours before the worst hour, across events.")


In [ ]:
# ============================================================================
# CELL D10 — SUMMARY AND DOWNLOAD (Part 2)
# ============================================================================
rows = []
for fp in sorted(glob.glob(os.path.join(RESULTS, "D*.json"))):
    r = json.load(open(fp)); rows.append({"test": r["test_id"], "status": r["status"], "kind": r["kind"]})
print(pd.DataFrame(rows).to_string(index=False))
zp = os.path.join(BASE, "depth_results.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in glob.glob(os.path.join(RESULTS, "D*.json")) + glob.glob(os.path.join(RESULTS, "futures_manifest_sha256.csv")) \
              + glob.glob(os.path.join(PREREG, "*depth*")):
        z.write(fp, os.path.relpath(fp, BASE))
print("zipped ->", zp)
if IN_COLAB:
    from google.colab import files
    files.download(zp)


## How to read the results
| Test | Supports T6 if | If not |
|---|---|---|
| D3 | C3 > 0 on most days, larger on high-volatility days | the estimated market is outside the amplification region; T6 stays theory |
| D4 | overshoot CI > 0 in (B) and much smaller in (C) | the overshoot is not carried by depth |
| D6 | C3 coefficient on liquidations > 0, CI above 0 (in-sample) | no link to forced selling |
| **D8** | Clark–West p < 0.05 and ΔR² CI > 0 on 2025–26 | report the null; T6 does not forecast at hourly horizon |
| D9 | C3 in a high percentile before the worst hours | events do not show the signature |

Send me `depth_results.zip` when it finishes.